# 🌊 Automated Flood Segmentation Pipeline (Google Colab)
### Chạy Tự Động Toàn Diện Cho 1 Model Trên Cả 2 Dataset & Tự Động Tải Về Zip

**Quy trình tự động hóa:**
1. Clone repository và cài đặt môi trường.
2. Huấn luyện đa seed (5 seeds: `42, 123, 456, 789, 2024`) trên **Dataset 1 (`floodkaggle`)**.
3. Huấn luyện đa seed (5 seeds) trên **Dataset 2 (`floodscene`)**.
4. Tự động xuất mô hình sang **ONNX** cho Jetson Orin Nano / Edge AI.
5. In bảng thống kê kết quả định dạng **LaTeX $(\mu \pm \sigma)$** phục vụ viết bài báo.
6. Nén toàn bộ kết quả (weights, logs, JSON, ONNX) thành file `.zip` và tự động tải về máy.

> 💡 **Cách dùng:** Bạn chỉ cần thay tên `MODEL_NAME` ở Cell 1 bên dưới rồi nhấn **Runtime -> Run all** (hoặc Ctrl+F9).

In [ ]:
# ==============================================================================
# ⚙️ CẤU HÌNH MÔ HÌNH CẦN CHẠY (BẠN CHỈ CẦN ĐỔI TÊN MODEL Ở ĐÂY)
# ==============================================================================
# Danh sách model có sẵn trong repo:
# - Mô hình chính: 'ours', 'mambafloodlite'
# - Baselines: 'segformer', 'fast-scnn', 'espnetv2', 'edanet', 'enet', 'stdc1', 'unet', 'segnet', 'unetformer', 'banet'
# - Ablations: 'abl_no_encoder_pfcu', 'abl_no_bottleneck', 'abl_relu', 'abl_kernel3', ...

MODEL_NAME = "ours"         # 👈 ĐỔI TÊN MODEL TẠI ĐÂY

# Các siêu tham số chuẩn thực nghiệm bài báo (A* / Q1)
EPOCHS = 50                 # Số epoch huấn luyện mỗi seed
BATCH_SIZE = 4              # Batch size tối ưu cho GPU T4/V100/A100
LR = 0.001                  # Tốc độ học (Learning Rate)
LOSS = "hard_afm"           # Hàm mất mát chuyên dụng cho ngập lụt
IMAGE_SIZE = 256            # Độ phân giải ảnh đầu vào (phải chia hết cho 16)
SEEDS = [42, 123, 456, 789, 2024] # 5 seeds đảm bảo tính tái lập nghiêm ngặt
DATASETS = ["floodkaggle", "floodscene"] # Chạy cả 2 bộ dữ liệu chuẩn

print(f"✓ Đã cấu hình xong cho Model: [{MODEL_NAME}]!")
print(f"✓ Sẽ chạy tự động trên 2 bộ: {DATASETS} với 5 seeds {SEEDS}.")


In [ ]:
# ==============================================================================
# 1. THIẾT LẬP MÔI TRƯỜNG & CLONE REPO (CHỐNG LỖI KHI CHẠY LẠI NHIỀU LẦN)
# ==============================================================================
import os

if not os.path.exists('Flood_Area_Segmentation') and not os.getcwd().endswith('Flood_Area_Segmentation'):
    !git clone https://github.com/doantrongthai/Flood_Area_Segmentation

if not os.getcwd().endswith('Flood_Area_Segmentation'):
    %cd Flood_Area_Segmentation

!git pull origin main
!pip install -r requirements.txt -q

print("\n✓ Môi trường đã sẵn sàng 100%!")


In [ ]:
# ==============================================================================
# 2. HUẤN LUYỆN TỰ ĐỘNG CẢ 2 BỘ DỮ LIỆU VỚI MULTI-SEED (5 SEEDS/BỘ)
# ==============================================================================
import os
import sys
import time

output_dir = f"outputs/{MODEL_NAME}"
os.makedirs(output_dir, exist_ok=True)
seeds_arg = " ".join(str(s) for s in SEEDS)

print("=" * 80)
print(f"🚀 BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN: MODEL = {MODEL_NAME}")
print("=" * 80)
total_start = time.time()

for idx, ds in enumerate(DATASETS, 1):
    print(f"\n{'#' * 80}")
    print(f"[{idx}/{len(DATASETS)}] BẮT ĐẦU CHẠY DATASET: {ds.upper()} ({len(SEEDS)} SEEDS x {EPOCHS} EPOCHS)")
    print(f"{'#' * 80}\n")
    
    cmd = (
        f"python benchmark.py "
        f"--dataset {ds} "
        f"--model {MODEL_NAME} "
        f"--size {IMAGE_SIZE} "
        f"--loss {LOSS} "
        f"--epochs {EPOCHS} "
        f"--batch_size {BATCH_SIZE} "
        f"--lr {LR} "
        f"--output_path {output_dir} "
        f"--multiseed "
        f"--seeds {seeds_arg} "
        f"--download"
    )
    
    ret = os.system(cmd)
    if ret != 0:
        print(f"❌ Có lỗi xảy ra khi chạy dataset {ds} (Exit code: {ret})!")
    else:
        print(f"✅ Hoàn thành xuất sắc toàn bộ {len(SEEDS)} seeds cho dataset {ds}!")

total_elapsed = (time.time() - total_start) / 60
print("\n" + "=" * 80)
print(f"🎉 HOÀN THÀNH TẤT CẢ DATASETS CHO MODEL: {MODEL_NAME} (Tổng thời gian: {total_elapsed:.1f} phút)")
print("=" * 80)


In [ ]:
# ==============================================================================
# 3. TỰ ĐỘNG XUẤT MÔ HÌNH SANG ONNX CHO BENCHMARK PHẦN CỨNG (JETSON / EDGE)
# ==============================================================================
import os

onnx_out = f"outputs/{MODEL_NAME}/{MODEL_NAME}_128x128.onnx"
print("=" * 80)
print(f"⚡ XUẤT MÔ HÌNH {MODEL_NAME} SANG ONNX TỐI ƯU HÓA PHẦN CỨNG EDGE")
print("=" * 80)

# Xuất ONNX (static resolution 128x128 cho Edge / Jetson benchmark)
ret_onnx = os.system(f"python export.py --model {MODEL_NAME} --size 128 --output {onnx_out}")
if ret_onnx == 0 and os.path.exists(onnx_out):
    size_kb = os.path.getsize(onnx_out) / 1024.0
    print(f"✓ Xuất ONNX thành công: {onnx_out} ({size_kb:.2f} KB)")
else:
    print(f"[INFO] Trạng thái xuất ONNX: {ret_onnx}. Có thể mô hình cần kernel chuyên dụng.")


In [ ]:
# ==============================================================================
# 4. TỔNG HỢP VÀ IN BẢNG BÁO CÁO KẾT QUẢ CHO BÀI BÁO (LATEX READY)
# ==============================================================================
import os
import glob
import json

print("\n" + "=" * 80)
print(f"📊 BÁO CÁO TỔNG HỢP KẾT QUẢ ĐA SEED CHO BÀI BÁO — MODEL: {MODEL_NAME.upper()}")
print("=" * 80)

json_files = sorted(glob.glob(f"outputs/{MODEL_NAME}/*_multiseed.json"))
if not json_files:
    json_files = sorted(glob.glob(f"outputs/*{MODEL_NAME}*_multiseed.json"))

for jf in json_files:
    with open(jf, 'r') as f:
        data = json.load(f)
    
    ds = data['config']['dataset'].upper()
    stats = data['statistics']
    comp = data.get('complexity', {})
    
    print(f"\n{'─' * 50}")
    print(f"📈 TẬP DỮ LIỆU: [{ds}] ({len(data['seeds'])} SEEDS)")
    print(f"{'─' * 50}")
    print(f"  • mIoU           : {stats['miou']['mean']:.4f} ± {stats['miou']['std']:.4f}")
    print(f"  • Dice (F1-score): {stats['dice']['mean']:.4f} ± {stats['dice']['std']:.4f}")
    print(f"  • Pixel Accuracy : {stats['pixel_accuracy']['mean']:.4f} ± {stats['pixel_accuracy']['std']:.4f}")
    print(f"  • Test Loss      : {stats['test_loss']['mean']:.4f} ± {stats['test_loss']['std']:.4f}")
    print(f"  • Tốc độ FPS     : {stats['fps']['mean']:.2f} ± {stats['fps']['std']:.2f}")
    print(f"  • Độ trễ Latency : {stats['latency_ms']['mean']:.2f} ± {stats['latency_ms']['std']:.2f} ms")
    if comp:
        print(f"  • Tổng Parameters: {comp.get('total_params', 0):,}")
        print(f"  • GFLOPs         : {comp.get('gflops', 0):.4f}")
        print(f"  • Model Size     : {comp.get('model_size_mb', 0):.2f} MB")
    
    print(f"\n  [LaTeX Code Copy Vào Bài Báo]:")
    print(f"  mIOU: ${stats['miou']['mean']:.4f} \\pm {stats['miou']['std']:.4f}$ | Dice: ${stats['dice']['mean']:.4f} \\pm {stats['dice']['std']:.4f}$ | FPS: ${stats['fps']['mean']:.2f} \\pm {stats['fps']['std']:.2f}$")
print("=" * 80)


In [ ]:
# ==============================================================================
# 5. TỰ ĐỘNG NÉN TOÀN BỘ KẾT QUẢ VÀ TẢI VỀ MÁY TÍNH (GOOGLE COLAB)
# ==============================================================================
import os
import shutil
from google.colab import files

target_folder = f"outputs/{MODEL_NAME}"
if not os.path.exists(target_folder):
    target_folder = "outputs"

zip_filename = f"{MODEL_NAME}_all_results"
zip_file = f"{zip_filename}.zip"

if os.path.exists(zip_file):
    os.remove(zip_file)

print("=" * 80)
print(f"📦 ĐANG NÉN TOÀN BỘ KẾT QUẢ CỦA MODEL '{MODEL_NAME}'...")
print("=" * 80)

shutil.make_archive(zip_filename, 'zip', target_folder)
file_size_mb = os.path.getsize(zip_file) / (1024 * 1024)

print(f"✓ Đã nén thành công: {zip_file} ({file_size_mb:.2f} MB)")
print(f"✓ File chứa đầy đủ: Checkpoints .pth từng seed, file JSON kết quả, và file ONNX.")
print("✓ Đang kích hoạt cửa sổ tải về máy tính qua trình duyệt...")

# Tải file về máy tính người dùng
files.download(zip_file)
print("🎉 ĐÃ HOÀN THÀNH TOÀN BỘ TIẾN TRÌNH!")
